# Entrainement - Plusieurs modeles d'incidents

Ce notebook entraine 3 modeles separes :

1. `load_spike_model` : predit `LOAD_SPIKE_ALL`.
2. `material_failure_model` : predit `CRASH_FAN` ou `THERMAL_DRIFT_SERVER`.
3. `incident_type_model` : modele multi-classe experimental (`none`, `LOAD_SPIKE_ALL`, `CRASH_FAN`, `THERMAL_DRIFT_SERVER`).

L'idee est d'eviter de melanger une surcharge globale avec des pannes materielles locales dans un seul label binaire.

## 0. Fichiers attendus

Le notebook attend ces fichiers a la racine du projet :

- `dataset_ml.csv`
- `fan_map.csv`

Si besoin, exporte-les avec :

```bash
docker compose -f docker-compose.yaml exec -T timescaledb \
  psql -U tsuser -d tsdb \
  -c "\COPY (
    SELECT
      sd.time,
      s.server_id,
      srv.hostname,
      MAX(CASE WHEN s.sensor_type = 'CPU_TEMP' THEN sd.value END) AS cpu_temp,
      MAX(CASE WHEN s.sensor_type = 'LOAD' THEN sd.value END) AS cpu_load,
      MAX(CASE WHEN s.sensor_type = 'TOTAL_POWER' THEN sd.value END) AS total_power,
      MAX(CASE WHEN s.sensor_type = 'FAN_SPEED_1' THEN sd.value END) AS fan_speed_1,
      MAX(CASE WHEN s.sensor_type = 'FAN_SPEED_2' THEN sd.value END) AS fan_speed_2
    FROM sensor_data sd
    JOIN sensor s ON s.sensor_id = sd.sensor_id
    JOIN server srv ON srv.server_id = s.server_id
    GROUP BY sd.time, s.server_id, srv.hostname
    ORDER BY sd.time, s.server_id
  ) TO STDOUT WITH CSV HEADER" > dataset_ml.csv

docker compose -f docker-compose.yaml exec -T timescaledb \
  psql -U tsuser -d tsdb \
  -c "\COPY (
    SELECT fan_id, server_id
    FROM fan
    ORDER BY fan_id
  ) TO STDOUT WITH CSV HEADER" > fan_map.csv
```

In [ ]:
# Si besoin
# %pip install pandas numpy matplotlib scikit-learn joblib

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, average_precision_score, precision_recall_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)

PROJECT_ROOT = Path("..").resolve()
DATASET_PATH = PROJECT_ROOT / "dataset_ml.csv"
FAN_MAP_PATH = PROJECT_ROOT / "fan_map.csv"
SCENARIO_PATH = PROJECT_ROOT / "nodejs-server" / "src" / "data_seed" / "scenarios.json"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

HORIZON_TICKS = 4
LOAD_SPIKE_THRESHOLD = 1.0  # value > 1.0 = vraie surcharge, value == 1.0 = retour a la normale

assert DATASET_PATH.exists(), f"Dataset introuvable: {DATASET_PATH}"
assert FAN_MAP_PATH.exists(), f"fan_map.csv introuvable: {FAN_MAP_PATH}"
assert SCENARIO_PATH.exists(), f"Scenario introuvable: {SCENARIO_PATH}"

## 1. Chargement et nettoyage

In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=["time"])
df.columns = [c.strip().lower() for c in df.columns]

df = (
    df.groupby(["time", "server_id", "hostname"], as_index=False)
      .agg({
          "cpu_temp": "mean",
          "cpu_load": "mean",
          "total_power": "mean",
          "fan_speed_1": "mean",
          "fan_speed_2": "mean",
      })
      .sort_values(["server_id", "time"])
      .reset_index(drop=True)
)

fan_map = pd.read_csv(FAN_MAP_PATH)
fan_map.columns = [c.strip().lower() for c in fan_map.columns]

print(df.shape)
display(df.head())
display(fan_map.head())

In [ ]:
time_steps = df[["time"]].drop_duplicates().sort_values("time")["time"].diff().dropna()
tick_delta = time_steps.mode().iloc[0]
start_time = df["time"].min()
print("start_time:", start_time)
print("tick_delta:", tick_delta)
print("end_time:", df["time"].max())

## 2. Construction des evenements par serveur

In [ ]:
with open(SCENARIO_PATH, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

scenario = next(s for s in scenarios if s["id"] == "sc_marseille_gpu_melt")
events_raw = pd.DataFrame(scenario["events"])

all_server_ids = sorted(df["server_id"].unique())

def resolve_server_ids(row):
    if row["type"] == "LOAD_SPIKE_ALL":
        return all_server_ids
    if row["type"] == "THERMAL_DRIFT_SERVER":
        return [int(row["targetId"])]
    if row["type"] == "CRASH_FAN":
        match = fan_map.loc[fan_map["fan_id"] == int(row["targetId"]), "server_id"]
        return [int(match.iloc[0])] if len(match) else []
    return []

events = events_raw[events_raw["type"].isin(["LOAD_SPIKE_ALL", "CRASH_FAN", "THERMAL_DRIFT_SERVER"])].copy()

# Important: dans scenarios.json, LOAD_SPIKE_ALL avec value == 1.0 signifie retour a la normale.
# Ce n'est pas une panne a predire, donc on l'exclut des labels positifs.
events = events[
    (events["type"] != "LOAD_SPIKE_ALL")
    | (events["value"].astype(float) > LOAD_SPIKE_THRESHOLD)
].copy()
events["server_id"] = events.apply(resolve_server_ids, axis=1)
events = events.explode("server_id").dropna(subset=["server_id"]).copy()
events["server_id"] = events["server_id"].astype(int)
events["event_time"] = start_time + events["tick"].astype(int) * tick_delta

display(events[["tick", "event_time", "type", "targetId", "server_id", "value"]].head(20))
display(events["type"].value_counts())

## 3. Labels separes

- `label_load_spike`: surcharge globale bientot.
- `label_material_failure`: panne materielle locale bientot.
- `label_incident_type`: type d'incident prioritaire bientot.

In [ ]:
labeled = df.copy()
labeled["label_load_spike"] = 0
labeled["label_material_failure"] = 0
labeled["label_incident_type"] = "none"

horizon_delta = HORIZON_TICKS * tick_delta
priority = {"none": 0, "LOAD_SPIKE_ALL": 1, "THERMAL_DRIFT_SERVER": 2, "CRASH_FAN": 3}

for event in events.itertuples(index=False):
    start_window = event.event_time - horizon_delta
    end_window = event.event_time
    mask = (
        (labeled["server_id"] == event.server_id)
        & (labeled["time"] >= start_window)
        & (labeled["time"] <= end_window)
    )
    if event.type == "LOAD_SPIKE_ALL":
        labeled.loc[mask, "label_load_spike"] = 1
    elif event.type in ["CRASH_FAN", "THERMAL_DRIFT_SERVER"]:
        labeled.loc[mask, "label_material_failure"] = 1

    current = labeled.loc[mask, "label_incident_type"]
    replace_mask = current.map(priority) < priority[event.type]
    labeled.loc[current.index[replace_mask], "label_incident_type"] = event.type

display(labeled["label_load_spike"].value_counts())
display(labeled["label_material_failure"].value_counts())
display(labeled["label_incident_type"].value_counts())

## 4. Features temporelles

In [ ]:
feature_df = labeled.copy()
sensor_cols = ["cpu_temp", "cpu_load", "total_power", "fan_speed_1", "fan_speed_2"]

for col in sensor_cols:
    group = feature_df.groupby("server_id")[col]
    feature_df[f"{col}_lag_1"] = group.shift(1)
    feature_df[f"{col}_lag_2"] = group.shift(2)
    feature_df[f"{col}_delta_1"] = group.diff(1)
    feature_df[f"{col}_delta_3"] = group.diff(3)
    feature_df[f"{col}_mean_3"] = group.transform(lambda s: s.rolling(3, min_periods=1).mean())
    feature_df[f"{col}_mean_6"] = group.transform(lambda s: s.rolling(6, min_periods=1).mean())
    feature_df[f"{col}_std_6"] = group.transform(lambda s: s.rolling(6, min_periods=2).std())

feature_df["fan_speed_mean"] = feature_df[["fan_speed_1", "fan_speed_2"]].mean(axis=1)
feature_df["fan_speed_min"] = feature_df[["fan_speed_1", "fan_speed_2"]].min(axis=1)
feature_df["temp_minus_fan"] = feature_df["cpu_temp"] - feature_df["fan_speed_mean"]
feature_df["power_per_load"] = feature_df["total_power"] / feature_df["cpu_load"].replace(0, np.nan)
feature_df["hour"] = feature_df["time"].dt.hour
feature_df["dayofweek"] = feature_df["time"].dt.dayofweek

feature_df = feature_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

label_cols = ["label_load_spike", "label_material_failure", "label_incident_type"]
drop_cols = ["time", "hostname", *label_cols]
feature_cols = [c for c in feature_df.columns if c not in drop_cols]

print(feature_df.shape)
print(len(feature_cols), "features")

## 5. Split temporel et fonction d'entrainement

In [ ]:
split_time = feature_df["time"].quantile(0.75)
train_df = feature_df[feature_df["time"] <= split_time].copy()
test_df = feature_df[feature_df["time"] > split_time].copy()

print("split_time:", split_time)
print("train:", train_df.shape, "test:", test_df.shape)

def candidate_models(binary=True):
    models = {
        "dummy": DummyClassifier(strategy="most_frequent"),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=10,
            min_samples_leaf=3,
            class_weight="balanced_subsample" if binary else "balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=400,
            max_depth=10,
            min_samples_leaf=3,
            class_weight="balanced" if binary else "balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "gradient_boosting": GradientBoostingClassifier(random_state=42),
    }
    if binary:
        models["logistic_regression"] = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced")
        )
    else:
        models["logistic_regression"] = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=3000, class_weight="balanced", multi_class="auto")
        )
    return models

def train_and_report(target_col, positive_label=1, binary=True):
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    print("Target:", target_col)
    print("Train distribution:")
    display(y_train.value_counts())
    print("Test distribution:")
    display(y_test.value_counts())

    rows = []
    trained = {}
    for name, model in candidate_models(binary=binary).items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        threshold = None
        pred_tuned = pred

        # Pour les labels desequilibres, le seuil 0.5 est rarement optimal.
        # On cherche le meilleur seuil F1 sur le train, puis on l'applique au test.
        if binary and hasattr(model, "predict_proba") and len(set(y_train)) == 2:
            train_proba = model.predict_proba(X_train)[:, 1]
            precision, recall, thresholds = precision_recall_curve(y_train, train_proba)
            f1 = (2 * precision * recall) / (precision + recall + 1e-12)
            if len(thresholds):
                best_idx = int(np.nanargmax(f1[:-1]))
                threshold = float(thresholds[best_idx])
                pred_tuned = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)

        report = classification_report(y_test, pred_tuned, output_dict=True, zero_division=0)
        trained[name] = model

        row = {"model": name, "accuracy": report.get("accuracy", 0), "threshold": threshold}
        if binary:
            key = str(positive_label)
            row.update({
                "precision_pos": report.get(key, {}).get("precision", 0),
                "recall_pos": report.get(key, {}).get("recall", 0),
                "f1_pos": report.get(key, {}).get("f1-score", 0),
            })
            if hasattr(model, "predict_proba"):
                row["average_precision"] = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])
            else:
                row["average_precision"] = np.nan
        else:
            row.update({
                "macro_f1": report.get("macro avg", {}).get("f1-score", 0),
                "weighted_f1": report.get("weighted avg", {}).get("f1-score", 0),
            })
        rows.append(row)

    results = pd.DataFrame(rows)
    sort_col = "f1_pos" if binary else "macro_f1"
    results = results.sort_values(sort_col, ascending=False)
    display(results)

    best_name = results.iloc[0]["model"]
    best_model = trained[best_name]
    pred = best_model.predict(X_test)
    best_threshold = results.iloc[0].get("threshold")
    if binary and hasattr(best_model, "predict_proba") and pd.notna(best_threshold):
        pred = (best_model.predict_proba(X_test)[:, 1] >= float(best_threshold)).astype(int)
    print("Best model:", best_name)
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
    plt.title(f"{target_col} - {best_name}")
    plt.show()
    return best_name, best_model, results

## 6. Modele 1 - LOAD_SPIKE_ALL

In [ ]:
load_spike_name, load_spike_model, load_spike_results = train_and_report("label_load_spike", binary=True)

## 7. Modele 2 - Pannes materielles

In [ ]:
material_name, material_model, material_results = train_and_report("label_material_failure", binary=True)

## 8. Modele 3 - Type d'incident multi-classe

Ce modele est surtout exploratoire. Les classes `CRASH_FAN` et `THERMAL_DRIFT_SERVER` sont rares, donc il risque d'etre moins stable.

In [ ]:
type_name, type_model, type_results = train_and_report("label_incident_type", binary=False)

## 9. Sauvegarde

In [ ]:
bundle = {
    "feature_columns": feature_cols,
    "horizon_ticks": HORIZON_TICKS,
    "tick_delta_seconds": tick_delta.total_seconds(),
    "models": {
        "load_spike": {"name": load_spike_name, "model": load_spike_model},
        "material_failure": {"name": material_name, "model": material_model},
        "incident_type": {"name": type_name, "model": type_model},
    },
    "results": {
        "load_spike": load_spike_results,
        "material_failure": material_results,
        "incident_type": type_results,
    },
}

out_path = MODEL_DIR / "incident_models.joblib"
joblib.dump(bundle, out_path)
out_path.resolve()

## 10. Exemple de prediction combinee

In [ ]:
sample = feature_df.sort_values("time").tail(20).copy()

sample["pred_load_spike"] = load_spike_model.predict(sample[feature_cols])
sample["pred_material_failure"] = material_model.predict(sample[feature_cols])
sample["pred_incident_type"] = type_model.predict(sample[feature_cols])

if hasattr(load_spike_model, "predict_proba"):
    sample["proba_load_spike"] = load_spike_model.predict_proba(sample[feature_cols])[:, 1]
if hasattr(material_model, "predict_proba"):
    sample["proba_material_failure"] = material_model.predict_proba(sample[feature_cols])[:, 1]

display(sample[[
    "time", "server_id", "hostname", "cpu_temp", "cpu_load", "fan_speed_mean",
    "label_load_spike", "label_material_failure", "label_incident_type",
    "pred_load_spike", "pred_material_failure", "pred_incident_type",
    *[c for c in ["proba_load_spike", "proba_material_failure"] if c in sample.columns]
]])